<div dir="rtl">

# 🏀 זריקה או מסירה – הפעלה בענן (חינם)

המחברת מפעילה את ממשק הניתוח על שרת של Google עם כרטיס גרפי (GPU), ונותנת קישור שאפשר לפתוח מכל מחשב או טלפון.

**פעם ראשונה בלבד: הרשאה לקוד**

המאגר ב-GitHub פרטי, ולכן Colab צריך מפתח גישה לקריאה:
1. ב-GitHub ← Settings ← Developer settings ← **Personal access tokens** ← **Fine-grained tokens** ← Generate new token.
   - Repository access: **Only select repositories** ← `guyos2/basketball`
   - Permissions ← Repository permissions ← **Contents: Read-only**
2. כאן ב-Colab, בסרגל הצד השמאלי, לוחצים על 🔑 (**Secrets**) ← Add new secret:
   - Name: `GITHUB_TOKEN`, Value: המפתח שיצרתם
   - מפעילים את **Notebook access**.

**בכל הפעלה**
1. **Runtime ← Change runtime type ← T4 GPU** ← Save.
2. מריצים את שני התאים למטה (▶). בסוף תאים 2 מופיעים קישור וסיסמה.
3. משאירים את הלשונית הזו פתוחה כל עוד עובדים. כשהיא נסגרת או אחרי זמן ללא שימוש, השרת נכבה והקבצים נמחקים. את התוצאות מורידים מעמוד התוצאות.

</div>

In [ ]:
#@title 1. התקנה (כ-2 דקות)
BRANCH = "claude/basketball-video-analysis-app-cv5xgk"  #@param {type:"string"}
import os, subprocess, sys

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception as e:  # secret missing or notebook access not granted
    print("לא נמצא GITHUB_TOKEN בסודות של Colab:", type(e).__name__)

repo = "/content/basketball"
url = f"https://{token}@github.com/guyos2/basketball.git" if token else "https://github.com/guyos2/basketball.git"
if not os.path.exists(repo):
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, repo], capture_output=True, text=True)
    if r.returncode:
        err = r.stderr.replace(token, "***") if token else r.stderr
        raise SystemExit("ההורדה מ-GitHub נכשלה. בדקו את GITHUB_TOKEN (הוראות למעלה).\n" + err)
else:
    subprocess.run(["git", "-C", repo, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", repo, "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{repo}/requirements.txt"], capture_output=True, text=True)
if r.returncode:
    raise SystemExit(r.stdout[-3000:] + r.stderr[-3000:])

import torch
if torch.cuda.is_available():
    print("✔ מוכן. GPU:", torch.cuda.get_device_name(0))
else:
    print("✔ מוכן, אבל בלי GPU – הניתוח יהיה איטי. Runtime ← Change runtime type ← T4 GPU, ואז להריץ שוב.")

In [ ]:
#@title 2. הפעלה – קבלת קישור
PASSWORD = ""  #@param {type:"string"}
import base64, os, re, secrets, subprocess, sys, time, urllib.request
from IPython.display import HTML, display

if not PASSWORD:
    PASSWORD = secrets.token_urlsafe(6)
env = dict(os.environ, COURTVISION_PASSWORD=PASSWORD,
           COURTVISION_MAX_UPLOAD_MB="95")  # the free Cloudflare tunnel accepts uploads up to 100MB
subprocess.run("pkill -f '[c]ourtvision serve'; pkill -f '[c]loudflared tunnel'", shell=True)

subprocess.Popen([sys.executable, "-m", "courtvision", "serve", "--host", "127.0.0.1", "--port", "8000",
                  "--workspace", "/content/workspace"],
                 cwd="/content/basketball", env=env, stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)

cloudflared = "/content/cloudflared"
if not os.path.exists(cloudflared):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
    os.chmod(cloudflared, 0o755)
subprocess.Popen([cloudflared, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                 stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)

auth = "Basic " + base64.b64encode(f"coach:{PASSWORD}".encode()).decode()
link = None
for _ in range(90):
    time.sleep(1)
    if link is None:
        found = re.search(r"https://[a-z0-9]+(?:-[a-z0-9]+)+\.trycloudflare\.com"  # tunnel names look like four-random-words, open("/content/tunnel.log").read())
        link = found.group(0) if found else None
    try:
        ok = urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8000/", headers={"Authorization": auth})).status == 200
    except Exception:
        ok = False
    if link and ok:
        break

if not ok:
    print(open("/content/server.log").read()[-3000:])
    raise SystemExit("השרת לא עלה – הפרטים למעלה.")
if not link:
    print(open("/content/tunnel.log").read()[-3000:])
    raise SystemExit("לא התקבל קישור מ-Cloudflare – נסו להריץ את התא שוב.")

display(HTML(f"""
<div dir="rtl" style="font-size:18px;line-height:1.8">
  <b>הממשק מוכן:</b> <a href="{link}" target="_blank">{link}</a><br>
  שם משתמש: כל שם &nbsp;·&nbsp; סיסמה: <code>{PASSWORD}</code><br>
  <span style="color:#888;font-size:14px">אפשר לפתוח את הקישור גם בטלפון. הקישור משתנה בכל הפעלה. השאירו את הלשונית הזו פתוחה.</span>
</div>"""))

In [ ]:
#@title 3. כיבוי (לא חובה)
import subprocess
subprocess.run("pkill -f '[c]ourtvision serve'; pkill -f '[c]loudflared tunnel'", shell=True)
print("כובה.")